# 03 — Portfolio Analysis

Multi-stock comparison, correlation analysis, and portfolio risk metrics.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.data_loader import StockDataLoader
from src.analysis import StockAnalyzer

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)

## Load Multiple Stocks

In [ ]:
loader = StockDataLoader(cache_dir='../data/raw')
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']
stocks = loader.fetch_multiple(tickers, start='2020-01-01')

print(f'Loaded {len(stocks)} stocks')
for t, d in stocks.items():
    print(f'  {t}: {len(d)} rows ({d.index.min().date()} to {d.index.max().date()})')

## Normalized Price Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
for ticker, df in stocks.items():
    normalized = df['Close'] / df['Close'].iloc[0] * 100
    ax.plot(df.index, normalized, label=ticker, linewidth=1.5)

ax.set_title('Normalized Price Comparison (Base = 100)', fontsize=14, fontweight='bold')
ax.set_ylabel('Normalized Price')
ax.legend()
plt.tight_layout()
plt.show()

## Correlation Matrix

In [ ]:
# Build returns DataFrame
returns_df = pd.DataFrame()
for ticker, df in stocks.items():
    returns_df[ticker] = df['Close'].pct_change()

corr = returns_df.corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            square=True, linewidths=1, ax=ax)
ax.set_title('Daily Returns Correlation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Individual Stock Stats

In [ ]:
stats_list = []
for ticker, df in stocks.items():
    analyzer = StockAnalyzer(df)
    stats = analyzer.summary_stats()
    stats['ticker'] = ticker
    stats_list.append(stats)

stats_df = pd.DataFrame(stats_list).set_index('ticker')
stats_df.style.format('{:.4f}').background_gradient(cmap='RdYlGn', axis=0)

## Risk vs Return

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(stats_df['annualized_volatility'] * 100,
           stats_df['annualized_return'] * 100,
           s=100, zorder=5)

for ticker in stats_df.index:
    ax.annotate(ticker,
                (stats_df.loc[ticker, 'annualized_volatility'] * 100,
                 stats_df.loc[ticker, 'annualized_return'] * 100),
                textcoords='offset points', xytext=(10, 5), fontsize=11)

ax.set_xlabel('Annualized Volatility (%)', fontsize=12)
ax.set_ylabel('Annualized Return (%)', fontsize=12)
ax.set_title('Risk vs Return', fontsize=14, fontweight='bold')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()